# Реакции пользователя

In [1]:
import numpy as np
import pandas as pd
import re
import os
from sklearn.pipeline import Pipeline
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score # для разделения с учетом групп
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from solution_utils import(MyModelReactions, read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, save_model, SEED)

In [2]:
DATA_PATH = "hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [3]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
gkf = GroupKFold(n_splits=5)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


Разделяем на train, val и test с учетом баланса классов и группировки

In [4]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.8125, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 65%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 15%
X_test, y_test = X[idx_test], y[idx_test] # 20%

groups_ids_train = groups_ids[idx_train_val[idx_train]]

In [5]:
for name, labels_part in [
    ("train", y_train),
    ("validation", y_val),
    ("test", y_test),
]:
    print(f"\n{name}: {len(labels_part)} примеров")
    print(
        pd.Series(labels_part)
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
        .round(3)
    )


train: 1954 примеров
positive    0.100
negative    0.317
neutral     0.583
Name: proportion, dtype: float64

validation: 449 примеров
positive    0.114
negative    0.303
neutral     0.584
Name: proportion, dtype: float64

test: 597 примеров
positive    0.094
negative    0.281
neutral     0.625
Name: proportion, dtype: float64


Получаем Tf-idf для всех данных

In [6]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

## Базовые модели

### Наивный Байес

Мы на семинаре использовали MultinomialNB, но сейчас больше подойдет ComplementNB, так как он специально спроектирован для несбалансированных данных, как у нас

In [7]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
rows = []
for alpha in alphas:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", ComplementNB(alpha=alpha)),
    ])
    scores = cross_val_score(
        pipe,
        X_train, y_train,
        cv=gkf, groups=groups_ids_train,
        scoring="f1_macro",
    )
    rows.append({
        "alpha": alpha,
        "mean_f1": round(scores.mean(), 4),
        "std_f1": round(scores.std(), 4),
    })

pd.DataFrame(rows).sort_values("mean_f1", ascending=False)

,alpha,mean_f1,std_f1
3,0.020,0.4821,0.0243
2,0.010,0.4785,0.0219
1,0.005,0.4777,0.0231
4,0.050,0.4662,0.0225
0,0.001,0.4646,0.0265
5,0.100,0.4398,0.0200
6,0.200,0.4051,0.0149
7,0.500,0.3716,0.0234
8,1.000,0.3436,0.0182


Берем alpha = 0.02

In [8]:
alpha_opt = 0.02
nb = ComplementNB(alpha=alpha_opt).fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_val_tfidf)

## Логистическая регрессия

In [9]:
C_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 1.5, 2.0, 5.0]
rows = []
for C in C_grid:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=5000, C=C, random_state=SEED)),  # веса классов обратно пропорционально частоте
    ])
    scores = cross_val_score(
        pipe,
        X_train, y_train,
        cv=gkf, groups=groups_ids_train,
        scoring="f1_macro",
    )
    rows.append({
        "C": C,
        "mean_f1": round(scores.mean(), 4),
        "std_f1":  round(scores.std(), 4),
    })

pd.DataFrame(rows).sort_values("mean_f1", ascending=False)

,C,mean_f1,std_f1
1,0.010,0.5556,0.0127
2,0.050,0.5552,0.0116
0,0.001,0.5551,0.0125
3,0.100,0.5548,0.0107
5,1.000,0.5524,0.0270
4,0.500,0.5498,0.0282
6,1.500,0.5485,0.0250
8,5.000,0.5476,0.0291
7,2.000,0.5459,0.0289


Выбираем C = 0.01

In [10]:
C_opt_logreg = 0.01
log_reg = LogisticRegression(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_logreg,
).fit(X_train_tfidf, y_train)
y_pred_logreg = log_reg.predict(X_val_tfidf)

## Метод опорных векторов

In [11]:
rows = []
for C in C_grid:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("clf", LinearSVC(class_weight="balanced", max_iter=5000, C=C, random_state=SEED)),
    ])
    scores = cross_val_score(
        pipe,
        X_train, y_train,
        cv=gkf, groups=groups_ids_train,
        scoring="f1_macro",
    )
    rows.append({
        "C": C,
        "mean_f1": round(scores.mean(), 4),
        "std_f1":  round(scores.std(), 4),
    })

pd.DataFrame(rows).sort_values("mean_f1", ascending=False)

,C,mean_f1,std_f1
4,0.500,0.5398,0.0308
7,2.000,0.5377,0.0314
5,1.000,0.5371,0.0328
8,5.000,0.5362,0.0299
6,1.500,0.5351,0.0305
3,0.100,0.4587,0.0306
2,0.050,0.4127,0.0127
0,0.001,0.2455,0.0028
1,0.010,0.2455,0.0028


Лучший С = 0.5

In [12]:
C_opt_svc = 0.5
svc = LinearSVC(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_svc,
).fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_val_tfidf)

In [13]:
metrics_models = []
metrics_models.append(calculate_metrics(y_val, y_pred_nb, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_logreg, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_svc, LABELS))

# baseline
dummy_most_frequent = DummyClassifier(strategy="most_frequent").fit(X_train_tfidf, y_train)
y_pred_dummy_mf = dummy_most_frequent.predict(X_val_tfidf)
metrics_models.append(calculate_metrics(y_val, y_pred_dummy_mf, LABELS))
dummy_stratified = DummyClassifier(strategy="stratified", random_state=0).fit(X_train_tfidf, y_train)
y_pred_dummy_st = dummy_stratified.predict(X_val_tfidf)
metrics_models.append(calculate_metrics(y_val, y_pred_dummy_st, LABELS))

metrics_models_df = pd.DataFrame(metrics_models, index=["NB", "LR", "SVC", "Dum_mf", "Dum_st"]).sort_values("macro-f1", ascending=False)
metrics_models_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
LR,0.630290,0.558362,0.554856,0.630919,"[[22, 10, 19], [14, 70, 52], [20, 51, 191]]","{'precision_positive': 0.39285714285714285, 'r..."
SVC,0.657016,0.498678,0.519767,0.629578,"[[10, 8, 33], [3, 60, 73], [4, 33, 225]]","{'precision_positive': 0.5882352941176471, 're..."
NB,0.630290,0.440752,0.428674,0.592472,"[[1, 13, 37], [4, 64, 68], [5, 39, 218]]","{'precision_positive': 0.1, 'recall_positive':..."
Dum_st,0.447661,0.345532,0.344749,0.447269,"[[9, 17, 25], [17, 36, 83], [28, 78, 156]]","{'precision_positive': 0.16666666666666666, 'r..."
Dum_mf,0.583519,0.333333,0.245663,0.430048,"[[0, 0, 51], [0, 0, 136], [0, 0, 262]]","{'precision_positive': 0.0, 'recall_positive':..."


Лучше всего себя показывает логистическая регрессия, берем ее за основу нашей модели и на ней проверяем гипотезы.

## Формируем "свою" модель на основе проверки гипотез о данных и параметрах

Изучим, как влияет выбор только первых/последних реплик на предсказания. Предположения: основная информация о реакции пользователя содержится в конце.

In [14]:
texts_by_variant = {
    "all": user_texts,
    "first": [
        clean_normalize_dialog(d, num=1, position='first')
        for d in labeled_dialogs
    ],
    "last": [
        clean_normalize_dialog(d, num=1, position='last')
        for d in labeled_dialogs
    ],
    "last_two": [
        clean_normalize_dialog(d, num=2, position='last')
        for d in labeled_dialogs
    ],
    "last_three": [
            clean_normalize_dialog(d, num=3, position='last')
            for d in labeled_dialogs
        ],
}

In [15]:
variant_results = []

for variant, texts in texts_by_variant.items():
    X_train_text = [texts[i] for i in idx_train_val[idx_train]]
    X_val_text   = [texts[i] for i in idx_train_val[idx_val]]

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
    )
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_val_vec   = vectorizer.transform(X_val_text)

    model = LogisticRegression(
        C=0.01,
        class_weight="balanced",
        max_iter=5000,
        random_state=SEED,
    )
    model.fit(X_train_vec, y_train)
    pred = model.predict(X_val_vec)

    m = calculate_metrics(y_val, pred, LABELS)

    variant_results.append({
        "variant":         variant,
        "macro-f1":        m["macro-f1"],
        "weighted-f1":     m["weighted-f1"],
        "accuracy":        m["accuracy"],
        "balanced_acc":    m["balanced_accuracy"],
        "F1_positive":     m["per_class_metrics"]["f1_positive"],
        "F1_neutral":      m["per_class_metrics"]["f1_neutral"],
        "F1_negative":     m["per_class_metrics"]["f1_negative"],
    })

variant_results_df = pd.DataFrame(variant_results).round(4).sort_values("macro-f1", ascending=False)

variant_results_df

,variant,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
3,last_two,0.5994,0.6571,0.6615,0.5850,0.5055,0.7417,0.5509
4,last_three,0.5907,0.6496,0.6503,0.5891,0.4950,0.7362,0.5407
0,all,0.5549,0.6309,0.6303,0.5584,0.4112,0.7290,0.5243
2,last,0.5325,0.6034,0.6147,0.5044,0.4167,0.7071,0.4737
1,first,0.4876,0.5808,0.5813,0.4871,0.2772,0.6781,0.5074


Ожидаемо, first — слабый, но, на удивление, last с одной репликой тоже плох. А вот 2 или 3 последние реплики уже лучше всего текста по большинству параметров.

Но нам этого недостаточно, плюс если мы совсем не будем учитывать остальной диалог, можем потерять длинные споры и недовольства в середине или (и так очень редкую) положительную реакцию. Попробуем просто добавить последним репликам веса.

Попробовали просто в hstack присоединить tfidf по последним строкам, умноженное на разные веса, но это почти ничего не дало. Не будем приводить здесь этот эксперимент, давайте попробуем добавить вес последним строкам иначе: отдельно обучаем 2 модели, одну на полных данных, вторую на последних репликах, и итоговые вероятности определяем как W * p_last + (1 - W) * p_full, p_last - вероятность, предсказанная по последним репликам, p_full - по всем

In [16]:
idx_train_real = idx_train_val[idx_train]
last2 = [texts_by_variant.get("last_two")[i] for i in idx_train_real]

fold_data = []  # здесь будут (y_val, p_full, p_last2, classes)

for fold_i, (tr_local, va_local) in enumerate(
    gkf.split(idx_train_real, y_train, groups=groups_ids_train)
):
    y_tr = y_train[tr_local]
    y_va = y_train[va_local]

    vec_full = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr_full = vec_full.fit_transform([X[i] for i in tr_local])
    X_va_full = vec_full.transform([X[i] for i in va_local])

    logreg_full = LogisticRegression(class_weight="balanced", C=0.01, max_iter=5000, random_state=SEED)
    logreg_full.fit(X_tr_full, y_tr)
    p_full = logreg_full.predict_proba(X_va_full)

    vec_l2 = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr_l2 = vec_l2.fit_transform([last2[i] for i in tr_local])
    X_va_l2 = vec_l2.transform([last2[i] for i in va_local])

    logreg_l2 = LogisticRegression(class_weight="balanced", C=0.01, max_iter=5000, random_state=SEED)
    logreg_l2.fit(X_tr_l2, y_tr)
    p_last2 = logreg_l2.predict_proba(X_va_l2)

    fold_data.append({
        "fold": fold_i,
        "y_true": y_va,
        "p_full": p_full,
        "p_last2": p_last2,
        "classes": logreg_full.classes_,
    })

In [17]:
from sklearn.metrics import f1_score
weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
rows = []

for W in weights:
    fold_scores = []
    for fd in fold_data:
        p_res = W * fd["p_last2"] + (1 - W) * fd["p_full"]
        pred = fd["classes"][p_res.argmax(axis=1)]
        fold_scores.append(f1_score(fd["y_true"], pred, average="macro"))
    rows.append({
        "W":       W,
        "CV_mean": round(np.mean(fold_scores), 4),
        "CV_std":  round(np.std(fold_scores), 4),
    })

cv_W = pd.DataFrame(rows).sort_values("CV_mean", ascending=False).reset_index(drop=True)
cv_W

,W,CV_mean,CV_std
0,0.8,0.5547,0.0311
1,0.9,0.5533,0.0272
2,0.7,0.5526,0.0272
3,1.0,0.5518,0.0303
4,0.6,0.5505,0.0325
5,0.5,0.5457,0.0285
6,0.4,0.5289,0.0261
7,0.3,0.5150,0.0280
8,0.2,0.4814,0.0364
9,0.1,0.3997,0.0319


По результатам валидации учший вес W = 0.8 для 2 последних строк. Причем вес 0 для последних строк - худший в данном эксперименте, и прирост идет по всем рассмотренным параметрам, особенно для нашего самого редкого позитивного класса! 

In [19]:
WEIGHT = 0.8

## Еще эксперименты с дополнительными метриками

Теперь попробуем добавить еще параметры, описывающие текст, и посмотреть, станет ли лучше.
В качестве новых параметров будем рассматривать:

1 эксперимент:
длина текста в символах
длина текста в словах
длина последних 2 реплик в символах
длина последних 2 реплик в словах
Предположение: не ожидаем особого прироста метрик

2 эксперимент:
количество ! в тексте
количество ? в тексте
количество идущих подряд знаков !!, ?? или !?
Предположение: может улучшить нахождение негативных реакций

3 эксперимент:
для заглавных букв в тексте
наличие слов из 2+ заглавных букв
Предположение: может улучшить нахождение негативных реакций

In [20]:
metrix_features = []
m = calculate_metrics(y_val, y_pred_logreg, labels=LABELS)
metrix_features.append({
            "macro-f1":      m["macro-f1"],
            "weighted-f1":   m["weighted-f1"],
            "accuracy":      m["accuracy"],
            "balanced_acc":  m["balanced_accuracy"],
            "F1_positive":   m["per_class_metrics"]["f1_positive"],
            "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
            "F1_negative":   m["per_class_metrics"]["f1_negative"],
        })

In [21]:
def features_lens(dialog):
    full_user = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    last2_user = "\n".join([t["text"] for t in dialog["turns"] if t["role"] == "user"][-2:])
    return [
        len(full_user),
        len(full_user.split()),
        len(last2_user),
        len(last2_user.split()),
    ]

EXCL = re.compile(r"!")
QUEST = re.compile(r"\?")
REPEAT = re.compile(r"!!|\?\?|!\?|\?!")

def features_puck(dialog):
    text = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    return [
        len(EXCL.findall(text)),
        len(QUEST.findall(text)),
        len(REPEAT.findall(text)),
    ]

CAPS_WORD = re.compile(r"\b[A-ZА-Я]{2,}\b")

def features_caps(dialog):
    text = "\n".join(t["text"] for t in dialog["turns"] if t["role"] == "user")
    letters = [l for l in text if l.isalpha()]
    if not letters:
        caps_ratio = 0
    else:  
        caps_ratio = sum(c.isupper() for c in letters) / len(letters)
    return [
        caps_ratio,
        int(bool(CAPS_WORD.search(text)))
    ]

In [22]:
def metrix_with_features(func):
    matrix_tr = csr_matrix([func(labeled_dialogs[i]) for i in idx_train_val[idx_train]])
    matrix_va = csr_matrix([func(labeled_dialogs[i]) for i in idx_train_val[idx_val]])

    # надо бы как-то нормировать наши признаки перед добавкой к общей матрице
    scaler = StandardScaler()
    F_tr_scaled = scaler.fit_transform(matrix_tr.toarray())
    F_va_scaled = scaler.transform(matrix_va.toarray())

    X_tr = hstack([X_train_tfidf, csr_matrix(F_tr_scaled)])
    X_va = hstack([X_val_tfidf, csr_matrix(F_va_scaled)])

    model = LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000, random_state=SEED).fit(X_tr, y_train)
    pred = model.predict(X_va)
    m = calculate_metrics(y_val, pred, labels=LABELS)
    return{
            "macro-f1":      m["macro-f1"],
            "weighted-f1":   m["weighted-f1"],
            "accuracy":      m["accuracy"],
            "balanced_acc":  m["balanced_accuracy"],
            "F1_positive":   m["per_class_metrics"]["f1_positive"],
            "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
            "F1_negative":   m["per_class_metrics"]["f1_negative"],
        }

In [23]:
metrix_features.append(metrix_with_features(features_lens))
metrix_features.append(metrix_with_features(features_puck))
metrix_features.append(metrix_with_features(features_caps))
metrix_features = pd.DataFrame(metrix_features, index=["base", "lens", "punct", "caps"]).sort_values("macro-f1", ascending=False)
metrix_features

,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
base,0.554856,0.630919,0.630290,0.558362,0.411215,0.729008,0.524345
punct,0.430128,0.537651,0.514477,0.440584,0.198758,0.657201,0.434426
lens,0.278956,0.308740,0.267261,0.374620,0.187166,0.323171,0.326531
caps,0.267796,0.246595,0.271715,0.404356,0.233216,0.168919,0.401254


Вывод: никакой видимой пользы от новых параметров не обнаружено.

## Сохраняем наши модели

сохраняем базовые модели

In [24]:
NB_PATH = os.path.join("artifacts", "reactions", "nb.joblib")
LR_PATH = os.path.join("artifacts", "reactions", "log_reg.joblib")
SVC_PATH = os.path.join("artifacts", "reactions", "svc.joblib")
DMF_PATH = os.path.join("artifacts", "reactions", "dmf.joblib")
DS_PATH = os.path.join("artifacts", "reactions", "ds.joblib")

save_model({"model": nb, "vectorizer": tfidf_vectorizer}, NB_PATH)
save_model({"model": log_reg, "vectorizer": tfidf_vectorizer}, LR_PATH)
save_model({"model": svc, "vectorizer": tfidf_vectorizer}, SVC_PATH)
save_model({"model": dummy_most_frequent, "vectorizer": tfidf_vectorizer}, DMF_PATH)
save_model({"model": dummy_stratified, "vectorizer": tfidf_vectorizer}, DS_PATH)

y_pred_dummy_mf = dummy_most_frequent.predict(X_test_tfidf)
y_pred_dummy_st = dummy_stratified.predict(X_test_tfidf)
y_pred_nb = nb.predict(X_test_tfidf)
y_pred_logreg = log_reg.predict(X_test_tfidf)
y_pred_svc = svc.predict(X_test_tfidf)

сохранено: artifacts\reactions\nb.joblib, 22137 КБ
сохранено: artifacts\reactions\log_reg.joblib, 13663 КБ
сохранено: artifacts\reactions\svc.joblib, 13663 КБ
сохранено: artifacts\reactions\dmf.joblib, 7307 КБ
сохранено: artifacts\reactions\ds.joblib, 7307 КБ


сохраняем свою модель

In [25]:
MY_MODEL_PATH = os.path.join("artifacts", "reactions", "mymodel.joblib")
labeled_dialogs = np.array(labeled_dialogs)
dialogs_train = labeled_dialogs[idx_train_val[idx_train]]
dialogs_test = labeled_dialogs[idx_test]

my_model = MyModelReactions(WEIGHT, C_opt_logreg).fit(dialogs_train, y_train)
y_pred_my = my_model.predict(dialogs_test)
save_model(my_model, MY_MODEL_PATH)

сохранено: artifacts\reactions\mymodel.joblib, 18085 КБ


In [26]:
metrics_models = []
metrics_models.append(calculate_metrics(y_test, y_pred_nb, LABELS))
metrics_models.append(calculate_metrics(y_test, y_pred_logreg, LABELS))
metrics_models.append(calculate_metrics(y_test, y_pred_svc, LABELS))
metrics_models.append(calculate_metrics(y_test, y_pred_dummy_mf, LABELS))
metrics_models.append(calculate_metrics(y_test, y_pred_dummy_st, LABELS))
metrics_models.append(calculate_metrics(y_test, y_pred_my, LABELS))

metrics_models_df = pd.DataFrame(metrics_models, index=["NB", "LR", "SVC", "Dum_mf", "Dum_st", "MyModel"]).sort_values("macro-f1", ascending=False)
metrics_models_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
MyModel,0.644891,0.561428,0.562946,0.645655,"[[24, 9, 23], [8, 88, 72], [21, 79, 273]]","{'precision_positive': 0.4528301886792453, 're..."
LR,0.636516,0.566380,0.548505,0.643396,"[[25, 5, 26], [15, 92, 61], [39, 71, 263]]","{'precision_positive': 0.31645569620253167, 'r..."
SVC,0.675042,0.511660,0.543967,0.655280,"[[14, 9, 33], [2, 74, 92], [3, 55, 315]]","{'precision_positive': 0.7368421052631579, 're..."
NB,0.644891,0.459466,0.468540,0.621595,"[[6, 9, 41], [3, 78, 87], [6, 66, 301]]","{'precision_positive': 0.4, 'recall_positive':..."
Dum_st,0.448911,0.345307,0.340310,0.458317,"[[11, 16, 29], [17, 46, 105], [50, 112, 211]]","{'precision_positive': 0.14102564102564102, 'r..."
Dum_mf,0.624791,0.333333,0.256357,0.480509,"[[0, 0, 56], [0, 0, 168], [0, 0, 373]]","{'precision_positive': 0.0, 'recall_positive':..."
